In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install awscli

import os

BASE = '/content/drive/MyDrive/Image-Text Retrieval'
os.makedirs(f'{BASE}/listings', exist_ok=True)
os.makedirs(f'{BASE}/images_meta', exist_ok=True)
os.makedirs(f'{BASE}/small_images', exist_ok=True)

folderlist = ['00','01','02','03','04','05','06','07','08','09','0a','0b','0c','0d','0e','0f','10','11','12',
              '13','14','15','16','17','18','19','1a','1b','1c','1d','1e','1f','20','21','22','23','24','25',
              '26','27','28','29','2a','2b','2c','2d','2e','2f','30','31','32','33','34','35','36','37','38',
              '39','3a','3b','3c','3d','3e','3f','40','41','42','43','44','45','46','47','48','49','4a','4b',
              '4c','4d','4e','4f','50','51','52','53','54','55','56','57','58','59','5a','5b','5c','5d','5e',
              '5f','60','61','62','63','64','65','66','67','68','69','6a','6b','6c','6d','6e','6f','70','71',
              '72','73','74','75','76','77','78','79','7a','7b','7c','7d','7e','7f','80','81','82','83','84',
              '85','86','87','88','89','8a','8b','8c','8d','8e','8f','90','91','92','93','94','95','96','97',
              '98','99','9a','9b','9c','9d','9e','9f','a0','a1','a2','a3','a4','a5','a6','a7','a8','a9','aa',
              'ab','ac','ad','ae','af','b0','b1','b2','b3','b4','b5','b6','b7','b8','b9','ba','bb','bc','bd',
              'be','bf','c0','c1','c2','c3','c4','c5','c6','c7','c8','c9','ca','cb','cc','cd','ce','cf','d0',
              'd1','d2','d3','d4','d5','d6','d7','d8','d9','da','db','dc','dd','de','df','e0','e1','e2','e3',
              'e4','e5','e6','e7','e8','e9','ea','eb','ec','ed','ee','ef','f0','f1','f2','f3','f4','f5','f6',
              'f7','f8','f9','fa','fb','fc','fd','fe','ff']
for folder in folderlist:
  os.makedirs(f'{BASE}/small_images/{folder}', exist_ok=True)


import pandas as pd
import gzip
import json
from collections import Counter
import subprocess
import numpy as np


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import subprocess

def download_if_missing(s3_path, local_path):
    if os.path.exists(local_path):
        print(f"already have {local_path}, skipping")
        return
    result = subprocess.run(
        ['aws', 's3', 'cp', '--no-sign-request', s3_path, local_path],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"FAILED: {s3_path}")
        print(result.stderr)
    else:
        print(f"downloaded: {local_path}")

def listings_gz_extract(file_name):
  data = []
  with gzip.open(f'{BASE}/listings/{file_name}', 'rt', encoding='utf-8') as f:
      print(f"opening {file_name}")
      for line in f:
          data.append(json.loads(line))

  file_df = pd.DataFrame(data)
  return file_df


def get_combined_files(all_files):
  columns_dict={}
  all_file_df = pd.DataFrame()
  for file_name in all_files:
    download_if_missing(
      f's3://amazon-berkeley-objects/listings/metadata/{file_name}',
      f'{BASE}/listings/{file_name}')

    file_df = listings_gz_extract(file_name)
    columns_dict[file_name] = file_df.columns

    all_file_df = pd.concat([all_file_df, file_df],ignore_index =True)
    print(f"appended {file_name}")

  return all_file_df

def download_if_missing_image_file(s3_path, local_path):
    if os.path.exists(local_path):
        print(f"already have {local_path}, skipping")
        return
    result = subprocess.run(
        ['aws', 's3', 'cp', '--no-sign-request', s3_path, local_path],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"FAILED: {s3_path}")
        print(result.stderr)
    else:
        print(f"downloaded: {local_path}")

def images_gz_extract(file_name):
  data = []
  file_df = pd.read_csv(f'{BASE}/images_meta/{file_name}', compression='gzip')
  return file_df
# ----   To Check wether all columns are identical or not in all the files ----
# all_cols = set()
# i=0
# for columns in columns_dict.values():
#   for column in columns:
#     all_cols.add(column)

# different_cols = {}
# for file,columns in columns_dict.items():
#   print(file, len(columns))
#   for column in columns:
#     if column not in all_cols:
#       l = []
#       l.append(column)
#       different_cols[file] = l
# print(f"not matched file and columns:  {different_cols}")

In [ ]:
!aws s3 ls --no-sign-request s3://amazon-berkeley-objects/images/metadata/

2021-06-17 09:06:19    6430535 images.csv.gz


In [ ]:
all_listings = [
'listings_0.json.gz',
'listings_1.json.gz',
'listings_2.json.gz',
'listings_3.json.gz',
'listings_4.json.gz',
'listings_5.json.gz',
'listings_6.json.gz',
'listings_7.json.gz',
'listings_8.json.gz',
'listings_9.json.gz',
'listings_a.json.gz',
'listings_b.json.gz',
'listings_c.json.gz',
'listings_d.json.gz',
'listings_e.json.gz',
'listings_f.json.gz']

combined_listing_df = get_combined_files(all_listings)
print(combined_listing_df.shape)

already have /content/drive/MyDrive/Image-Text Retrieval/listings/listings_0.json.gz, skipping
opening listings_0.json.gz
appended listings_0.json.gz
already have /content/drive/MyDrive/Image-Text Retrieval/listings/listings_1.json.gz, skipping
opening listings_1.json.gz
appended listings_1.json.gz
already have /content/drive/MyDrive/Image-Text Retrieval/listings/listings_2.json.gz, skipping
opening listings_2.json.gz
appended listings_2.json.gz
already have /content/drive/MyDrive/Image-Text Retrieval/listings/listings_3.json.gz, skipping
opening listings_3.json.gz
appended listings_3.json.gz
already have /content/drive/MyDrive/Image-Text Retrieval/listings/listings_4.json.gz, skipping
opening listings_4.json.gz
appended listings_4.json.gz
already have /content/drive/MyDrive/Image-Text Retrieval/listings/listings_5.json.gz, skipping
opening listings_5.json.gz
appended listings_5.json.gz
already have /content/drive/MyDrive/Image-Text Retrieval/listings/listings_6.json.gz, skipping
openi

In [ ]:
combined_listing_df[combined_listing_df['item_id']=='B06X9STHNG']

,brand,bullet_point,color,item_id,item_name,model_name,model_number,model_year,product_type,style,...,item_weight,material,fabric_type,color_code,product_description,spin_id,3dmodel_id,pattern,finish_type,item_shape
0,"[{'language_tag': 'nl_NL', 'value': 'find.'}]","[{'language_tag': 'nl_NL', 'value': 'Schoen in...","[{'language_tag': 'nl_NL', 'value': 'Veelkleur...",B06X9STHNG,"[{'language_tag': 'nl_NL', 'value': 'Amazon-me...","[{'language_tag': 'nl_NL', 'value': '37753'}]",[{'value': '12-05-04'}],[{'value': 2017}],[{'value': 'SHOES'}],"[{'language_tag': 'nl_NL', 'value': 'Gesloten-...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
List_df = combined_listing_df.copy()

# Remove greater than 70% null rate columns

data = combined_listing_df.isnull().sum() * 100 / combined_listing_df.shape[0]
null_rates = pd.DataFrame(data,columns=['null_rate']).reset_index()
cols_to_drop = list(null_rates.loc[null_rates['null_rate'] > 60,'index'])

cols_to_drop.extend(['model_number','other_image_id','domain_name','item_weight'])
List_df = List_df.drop(columns=cols_to_drop)
print(f"Total remove columns are {len(cols_to_drop)} and remaining columns are {List_df.columns}")

# Drop the values with multiple item ids. It introduces unwanted complications,
# and total 1932 item ids have duplicates. Reasonable to Drop
exp_data = List_df[['item_id','country']]

exp_data2 = (
    exp_data.groupby('item_id')
    .agg(
        count=('item_id', 'count'),
        countries=('country', list)
    )
    .sort_values(by='count', ascending=False)
)
before_count = List_df.shape[0]
duplicate_items = exp_data2.loc[exp_data2['count'] > 1].index
List_df = List_df[~List_df['item_id'].isin(duplicate_items)]

removed_rows = before_count - List_df.shape[0]

print(f"Total removed duplicated values are {removed_rows}")


# Can also drop marketplace and Node and Country. will do after further EDA

Total remove columns are 16 and remaining columns are Index(['brand', 'bullet_point', 'color', 'item_id', 'item_name', 'model_name',
       'product_type', 'main_image_id', 'item_keywords', 'country',
       'marketplace', 'node'],
      dtype='object')
Total removed duplicated values are 4019


In [ ]:
List_df.head()

,brand,bullet_point,color,item_id,item_name,model_name,product_type,main_image_id,item_keywords,country,marketplace,node
0,"[{'language_tag': 'nl_NL', 'value': 'find.'}]","[{'language_tag': 'nl_NL', 'value': 'Schoen in...","[{'language_tag': 'nl_NL', 'value': 'Veelkleur...",B06X9STHNG,"[{'language_tag': 'nl_NL', 'value': 'Amazon-me...","[{'language_tag': 'nl_NL', 'value': '37753'}]",[{'value': 'SHOES'}],81iZlv3bjpL,"[{'language_tag': 'nl_NL', 'value': 'block hee...",NL,Amazon,"[{'node_id': 16391787031, 'node_name': '/Categ..."
1,"[{'language_tag': 'es_MX', 'value': 'AmazonBas...","[{'language_tag': 'es_MX', 'value': 'White Pow...","[{'language_tag': 'es_MX', 'value': 'White Pow...",B07P8ML82R,"[{'language_tag': 'es_MX', 'value': '22"" Botto...",NaN,[{'value': 'HARDWARE'}],619y9YG9cnL,"[{'language_tag': 'es_MX', 'value': '22'}, {'l...",MX,Amazon,"[{'node_id': 9827962011, 'node_name': '/Catego..."
2,"[{'language_tag': 'en_AE', 'value': 'AmazonBas...","[{'language_tag': 'en_AE', 'value': '3D printe...","[{'language_tag': 'en_AE', 'value': 'Transluce...",B07H9GMYXS,"[{'language_tag': 'en_AE', 'value': 'AmazonBas...",NaN,[{'value': 'MECHANICAL_COMPONENTS'}],81NP7qh2L6L,"[{'language_tag': 'en_AE', 'value': '3d printe...",AE,Amazon,"[{'node_id': 11601270031, 'node_name': '/Categ..."
3,"[{'language_tag': 'en_GB', 'value': 'Stone & B...",NaN,"[{'language_tag': 'en_GB', 'value': 'Stone Bro...",B07CTPR73M,"[{'language_tag': 'en_GB', 'value': 'Stone & B...",NaN,[{'value': 'SOFA'}],61Rp4qOih9L,"[{'language_tag': 'en_GB', 'value': 'love'}, {...",GB,Amazon,"[{'node_id': 2850919031, 'node_name': '/Home &..."
4,"[{'language_tag': 'en_AU', 'value': 'The Fix'}]","[{'language_tag': 'en_AU', 'value': 'Embroider...","[{'language_tag': 'en_AU', 'standardized_value...",B01MTEI8M6,"[{'language_tag': 'en_AU', 'value': 'The Fix A...",NaN,[{'value': 'SHOES'}],714CmIfKIYL,"[{'language_tag': 'en_AU', 'value': 'zapatos s...",AU,Amazon,"[{'node_id': 5131142051, 'node_name': '/Catego..."


In [ ]:
exp_data3_cntry = exp_data.groupby(by='country').agg(count=('country','count')).sort_values(by='count',ascending=False) / exp_data.shape[0]
exp_data3_cntry = exp_data3_cntry.reset_index()
exp_data3_cntry['cum_sum'] = exp_data3_cntry['count'].cumsum()
exp_data3_cntry

lang_counts = Counter()
for file_name in all_listings:
  with gzip.open(f'{BASE}/listings/{file_name}', 'rt', encoding='utf-8') as f:
      for line in f:
          raw_row = json.loads(line)
          brand_value = raw_row.get('brand')

          if not isinstance(brand_value, list):
              continue  # skip rows where brand is missing/NaN entirely

          for pair in brand_value:
              tag = pair.get('language_tag')
              if tag:
                  lang_counts[tag] += 1

lang_counts.most_common()

exp_data3_brnd = pd.DataFrame.from_dict(lang_counts, orient='index', columns=['Count'])
exp_data3_brnd['Count'] = exp_data3_brnd['Count'] / exp_data3_brnd['Count'].sum()
exp_data3_brnd = exp_data3_brnd.reset_index().rename(columns={'index':'brand_lang'}).sort_values(by='Count',ascending=False)
exp_data3_brnd['cum_sum'] = exp_data3_brnd['Count'].cumsum()
exp_data3_brnd

# This was used to check the language and contries distribution before taking action
# to select onlt english language codes
# Choosen codes
# english_tags = {'en_IN', 'en_US', 'en_CA', 'en_GB', 'en_AU', 'en_AE', 'en_SG'}

,country,count,cum_sum
0,IN,0.517542,0.517542
1,US,0.167899,0.685441
2,CA,0.045754,0.731195
3,GB,0.036817,0.768013
4,DE,0.032816,0.800829
5,MX,0.032708,0.833536
6,ES,0.030622,0.864159
7,FR,0.028124,0.892283
8,IT,0.026662,0.918945
9,JP,0.025233,0.944178


In [ ]:
List_df['node'].isnull().sum()

np.int64(6777)

In [ ]:
data = []

for file_name in all_listings:
  with gzip.open(f'{BASE}/listings/{file_name}', 'rt', encoding='utf-8') as f:
      for line in f:
          raw_row = json.loads(line)
          mod_row = {}

          key_value = raw_row.get('node')
          if not isinstance(key_value, list):
              continue
          for pair in key_value:
            mod_row['node'] = pair.get('node_name')

          data.append(mod_row)

node_df = pd.DataFrame(data)

# node_df['node_list'] = node_df['node'].str.split('/')
# node_df['length'] = node_df['node_list'].apply(lambda x: 0 if x is None else len(x)-1 )
# node_df.sort_values(by='length',ascending=False)

# node_df.groupby(by='length').agg(Count=('length','count')).reset_index()

# Based on the analysis let's keep Node for now later on it might be useful as additional detail to product type

,node
0,/Categorieën/Dames/Schoenen/Pumps
1,/Categorías/Ferretería/Ferretería para Armario...
2,/Categories
3,/Home & Garden/Home & Kitchen/Categories/Furni...
4,/Categories/Women/Shoes/Loafer Flats
...,...
140744,/Categorías/Pequeño electrodoméstico/Microonda...
140745,/Categories/Mobiles & Accessories/Mobile Acces...
140746,/Categories/Pantry Staples/Dried Grains & Rice...
140747,"/Categorías/Decoración del Hogar/Alfombras, Al..."


In [ ]:
node_class = node_df['node'].apply(lambda x: type('any string') if x is None else type(x))
node_class[node_class != str]

,node


In [ ]:
def extract_node_field(raw_row, key='node', join_multi=False):
    key_value = raw_row.get(key)
    if not isinstance(key_value, list) or len(key_value) == 0:
        return ''
    names = [pair.get('node_name', '') for pair in key_value if pair.get('node_name')]
    if not names:
        return ''
    return ' | '.join(names) if join_multi else names[0]

for file_name in all_listings:
  with gzip.open(f'{BASE}/listings/{file_name}', 'rt', encoding='utf-8') as f:
    for line in f:
        raw_row = json.loads(line)
        mod_row = {}

        mod_row['node'] = extract_node_field(raw_row)

    data.append(mod_row)

node_df2 = pd.DataFrame(data)
node_df2

,node
0,/Categorieën/Dames/Schoenen/Pumps
1,/Categorías/Ferretería/Ferretería para Armario...
2,/Categories
3,/Home & Garden/Home & Kitchen/Categories/Furni...
4,/Categories/Women/Shoes/Loafer Flats
...,...
140770,/Categories/Kitchen & Home Appliances/Small Ki...
140771,/Categorías/Mujeres/Zapatos/Botas
140772,"/Categories/Beverages/Coffee, Tea & Cocoa/Coff..."
140773,/Categories/Mobiles & Accessories/Mobile Acces...


In [ ]:
node_df2['node_list'] = node_df2['node'].str.split('/')
node_df2['length'] = node_df2['node_list'].apply(lambda x: 0 if x is None else len(x)-1 )
node_df2.sort_values(by='length',ascending=False)

# node_df2.groupby(by='length').agg(Count=('length','count')).reset_index()


,node,node_list,length
116798,/Home & Garden/Home & Kitchen/Categories/Cooki...,"[, Home & Garden, Home & Kitchen, Categories, ...",9
37829,/Home & Garden/Home & Kitchen/Categories/Cooki...,"[, Home & Garden, Home & Kitchen, Categories, ...",9
62930,/Home & Garden/Home & Kitchen/Categories/Cooki...,"[, Home & Garden, Home & Kitchen, Categories, ...",8
51357,/Home & Garden/Home & Kitchen/Categories/Cooki...,"[, Home & Garden, Home & Kitchen, Categories, ...",8
89678,/Home & Garden/Home & Kitchen/Categories/Cooki...,"[, Home & Garden, Home & Kitchen, Categories, ...",8
...,...,...,...
45421,None,None,0
45753,None,None,0
54672,None,None,0
47344,None,None,0


In [ ]:
import gzip
import json

def extract_value_only(raw_row, key):
    # for columns like product_type, model_number: list of {'value': ...}, no language_tag at all
    key_value = raw_row.get(key)
    if not isinstance(key_value, list) or len(key_value) == 0:
        return ''
    return key_value[0].get('value', '')

value_only_cols = ['product_type']

data = []

for file_name in all_listings:
  with gzip.open(f'{BASE}/listings/{file_name}', 'rt', encoding='utf-8') as f:
    for line in f:
        raw_row = json.loads(line)
        mod_row = {}

        for col in value_only_cols:
            mod_row[col] = extract_value_only(raw_row, col)

        data.append(mod_row)

product_df = pd.DataFrame(data)
product_type_df = product_df.copy()

In [ ]:
with pd.option_context('display.max_colwidth', None):
    print(combined_listing_df[['item_keywords']].head(5))


In [ ]:
## Get All the Files Data with Specific Languages and Column level logics applied
# Based on previous data exploration
import gzip
import json
import pandas as pd

english_tags = {'en_IN', 'en_US', 'en_CA', 'en_GB', 'en_AU', 'en_AE', 'en_SG'}


def extract_lang_field(raw_row, key, language_codes, join_multi=False):
    key_value = raw_row.get(key)
    if not isinstance(key_value, list):
        return ''
    matches = [pair['value'] for pair in key_value if pair.get('language_tag') in language_codes]
    if not matches:
        return ''
    if join_multi:
        matches = list(dict.fromkeys(matches))  # dedupe, preserves first-seen order
    return ' | '.join(matches) if join_multi else matches[0]

def extract_value_only(raw_row, key):
    # for columns like product_type, model_number: list of {'value': ...}, no language_tag at all
    key_value = raw_row.get(key)
    if not isinstance(key_value, list) or len(key_value) == 0:
        return ''
    return key_value[0].get('value', '')

def extract_raw(raw_row, key):
    # for plain scalar columns: item_id, country, marketplace, domain_name, main_image_id
    value = raw_row.get(key)
    return value if value is not None else ''

def extract_node_field(raw_row, key='node', join_multi=False):
    key_value = raw_row.get(key)
    if not isinstance(key_value, list) or len(key_value) == 0:
        return ''
    names = [pair.get('node_name', '') for pair in key_value if pair.get('node_name')]
    if not names:
        return ''
    return ' | '.join(names) if join_multi else names[0]

language_cols = {
    'brand': False,
    'bullet_point': True,
    'color': False,
    'item_name': False,
    'model_name': False,
    'item_keywords': True,
}
value_only_cols = ['product_type']
raw_cols = ['item_id', 'country', 'marketplace', 'main_image_id']

all_listings = [
'listings_0.json.gz',
'listings_1.json.gz',
'listings_2.json.gz',
'listings_3.json.gz',
'listings_4.json.gz',
'listings_5.json.gz',
'listings_6.json.gz',
'listings_7.json.gz',
'listings_8.json.gz',
'listings_9.json.gz',
'listings_a.json.gz',
'listings_b.json.gz',
'listings_c.json.gz',
'listings_d.json.gz',
'listings_e.json.gz',
'listings_f.json.gz']

data = []
for file_name in all_listings:
  with gzip.open(f'{BASE}/listings/{file_name}', 'rt', encoding='utf-8') as f:
      for line in f:
          raw_row = json.loads(line)
          mod_row = {}

          for col, multi in language_cols.items():
              mod_row[col] = extract_lang_field(raw_row, col, english_tags, join_multi=multi)

          for col in value_only_cols:
              mod_row[col] = extract_value_only(raw_row, col)

          for col in raw_cols:
              mod_row[col] = extract_raw(raw_row, col)

          mod_row['node'] = extract_node_field(raw_row, key='node', join_multi=False)

          data.append(mod_row)

df_final = pd.DataFrame(data)



In [ ]:
brands_df = df_final.copy()
brands_df = brands_df.replace('', np.nan)

#  Removing these rows as they are not important at any stage
cols = ['brand','bullet_point','color','item_name','model_name','item_keywords']
brands_df = brands_df[~brands_df[cols].isna().all(axis=1)]

brands_df['combined_text'] = brands_df['item_name'] + " " + brands_df['bullet_point']
brands_df = brands_df[~brands_df['combined_text'].isna()]
brands_df = brands_df.drop(columns=['item_name', 'bullet_point'])
brands_df['ct_len'] = brands_df['combined_text'].apply(len)

# Priority List
country_priority = ['IN', 'US', 'CA', 'GB', 'DE', 'MX', 'ES', 'FR', 'IT', 'JP',
                     'AU', 'NL', 'SG', 'AE', 'SE', 'SA', 'TR', 'UK', 'PL', 'BR']
priority_map = {c: i for i, c in enumerate(country_priority)}
brands_df['country_rank'] = brands_df['country'].map(priority_map).fillna(len(country_priority))


def dedup_by_group(df, group_col):
    dup_mask = df.groupby(group_col)[group_col].transform('count') > 1
    dup_subset = df[dup_mask].sort_values(
        by=[group_col, 'ct_len', 'country_rank'],
        ascending=[True, False, True]
    )
    winners = dup_subset.drop_duplicates(subset=group_col, keep='first')
    losing_indices = dup_subset.index.difference(winners.index)
    return df.drop(index=losing_indices), len(losing_indices)


brands_df, n_dropped_item = dedup_by_group(brands_df, 'item_id')
print(f"dropped {n_dropped_item} losing rows from item_id duplicates")
assert brands_df['item_id'].duplicated().sum() == 0



brands_df, n_dropped_image = dedup_by_group(brands_df, 'main_image_id')
print(f"dropped {n_dropped_image} losing rows from main_image_id duplicates")

before = len(brands_df)
brands_df = brands_df[~brands_df['main_image_id'].isna()]
print(f"dropped {before - len(brands_df)} rows with no main_image_id")

assert brands_df['main_image_id'].duplicated().sum() == 0

product_df = brands_df['product_type'].value_counts().reset_index()
new_product_df = product_df.loc[(product_df['count']>300) & (product_df['product_type']!='CELLULAR_PHONE_CASE'),'product_type']

brands_df = brands_df[brands_df['product_type'].isin(new_product_df)] # 25479 rows


dropped 459 losing rows from item_id duplicates
dropped 9670 losing rows from main_image_id duplicates
dropped 277 rows with no main_image_id


In [ ]:
brands_df

,brand,color,model_name,item_keywords,product_type,item_id,country,marketplace,main_image_id,node,combined_text,ct_len,country_rank
4,The Fix,Havana Tan,NaN,zapatos shoe para de ladies mujer womans mocas...,SHOES,B01MTEI8M6,AU,Amazon,714CmIfKIYL,/Categories/Women/Shoes/Loafer Flats,The Fix Amazon Brand Women's French Floral Emb...,226,10
35,Stanton,Brown,NaN,Stanton | Brown | 8 UK (42 EU) (9 US) | FK/YT1...,SHOES,B0846BTJ34,IN,Amazon,71kE0ihplPL,/Categories/Shoes/Men's Shoes/Casual Shoes/Loa...,Stanton Men's Brown Loafers-8 UK (42 EU) (9 US...,191,0
39,The Fix,Taupe,NaN,zapatos shoe para de ladies mujer womans | des...,SHOES,B01N27SMXC,US,Amazon,71-eMu0gN+L,/Departments/Women/Shops,Amazon Brand - The Fix Women's Foley Tassel Sl...,199,1
50,206 Collective,Black Suede,NaN,fathers day gifts for dad | party | ladies pum...,SHOES,B01N5XROXR,AE,Amazon,61uONZVpAEL,/Categories/Men/Shoes/Loafer Flats,Amazon Brand - 206 Collective Men's Pike Drivi...,195,13
55,365 Everyday Value,NaN,NaN,NaN,GROCERY,B07QC4DZ2L,US,WholeFoods,81oNfRT6ebL,"/Categories/Dairy, Cheese & Eggs/Milk & Cream/...","365 EVERYDAY VALUE Organic Light Cream, 1 PT B...",435,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
147666,365 Everyday Value,NaN,NaN,NaN,GROCERY,B07SY3YFC1,US,WholeFoods,71MgCNGNF+L,NaN,"365 EVERYDAY VALUE Original Oatmilk, 64 FZ Bro...",576,1
147671,AmazonBasics,Cool White,NaN,This bulb is energy star rated and UL listed |...,LIGHT_BULB,B079VNBKJB,US,Amazon,71oRw-drx5L,/Categories/Light Bulbs/LED Bulbs,"AmazonBasics Commercial Grade 25,000 Hour LED ...",893,1
147685,Flavia,GREY,NaN,NaN,SHOES,B0845BV6H6,IN,Amazon,811AS57BWVL,/Categories/Shoes/Women's Shoes/Sports & Outdo...,FLAVIA Women's Grey Running Shoes-UK6 (FKT/SD-...,187,0
147686,Unknown,Opulence Grey,NaN,Pinzon by Amazon | Supima cotton;bathroom;towe...,HOME,B003JFJWVG,CA,Amazon,91IqWKj2vQL,NaN,"Pinzon Oversized Supima Cotton Wash Cloth, Opp...",363,2


In [ ]:
brands_df['main_image_id'].duplicated().sum()

np.int64(0)

In [ ]:
brands_df['product_type'].value_counts().head(15)

,count
product_type,
CELLULAR_PHONE_CASE,64757
SHOES,9970
GROCERY,6214
HOME,2642
CHAIR,1714
HOME_BED_AND_BATH,1664
HOME_FURNITURE_AND_DECOR,1435
HEALTH_PERSONAL_CARE,1207
BOOT,1161


In [ ]:
removed_df['product_type'].value_counts().head(15)

,count
product_type,
SHOES,522
FINERING,104
BOOT,104
GROCERY,103
SANDAL,87
HOME_BED_AND_BATH,81
HOME,65
FINENECKLACEBRACELETANKLET,52
FINEEARRING,34


In [ ]:
removed_df.shape

(910, 13)

In [ ]:
item_cntry_grp = removed_df.groupby('item_id').agg(Count = ('country','count')).reset_index()


removed_df[removed_df['item_id'].isin(item_cntry_grp.loc[item_cntry_grp['Count']!=2,'item_id'])].sort_values(by='item_id')

,brand,bullet_point,color,item_name,model_name,item_keywords,product_type,item_id,country,marketplace,main_image_id,node
147438,NaN,Battery with improved low self-discharge. | Im...,NaN,AmazonBasics Pre-Charged Ni-MH Batteries,NaN,NaN,BATTERY,B00EZ1ZTFG,DE,Amazon,61kYGJ9rKHL,"/Kategorien/Batterien, Akkus & Zubehör/Akkus"
115650,AmazonBasics,An Amazon Brand,NaN,AmazonBasics Ni-MH Pre-Charged Rechargeable Ba...,NaN,NaN,BATTERY,B00EZ1ZTFG,US,PrimeNow,6186EOdL8eL,/Categories
51960,AmazonBasics,An Amazon Brand,NaN,AmazonBasics Ni-MH Pre-Charged Rechargeable Ba...,NaN,NaN,BATTERY,B00EZ1ZTFG,SG,Amazon,71HSrPIs-sL,/Categories/Household Batteries & Chargers
103504,AmazonBasics,An Amazon Brand,NaN,AmazonBasics 16/3 Vinyl Outdoor Extension Cord...,NaN,NaN,HOME_LIGHTING_ACCESSORY,B00TIFV5BG,AU,Amazon,81DT3WFI4ZL,/Categories
66240,AmazonBasics,An Amazon Brand,NaN,AmazonBasics 16/3 Vinyl Outdoor Extension Cord...,NaN,NaN,HOME_LIGHTING_ACCESSORY,B00TIFV5BG,CA,Amazon,813Dn19M6RL,/Categories/Electrical/Extension Cords
79303,AmazonBasics,An Amazon Brand,NaN,AmazonBasics 16/3 Vinyl Outdoor Extension Cord,NaN,NaN,HOME_LIGHTING_ACCESSORY,B00TIFV5BG,US,PrimeNow,81ompRLEmoL,/Categories
4741,NaN,Amazon Brand,NaN,AmazonBasics Reversible Microfiber Comforter,NaN,NaN,HOME_BED_AND_BATH,B00U8QGU32,JP,Amazon,710i81Pb9tL,/カテゴリー別/寝具/掛けふとん
106356,AmazonBasics,An Amazon Brand,NaN,AmazonBasics Reversible Microfiber Comforter B...,NaN,bed | pink | queen | navy | goose | canadian |...,HOME_BED_AND_BATH,B00U8QGU32,US,Amazon,6165OiMTufL,/Categories/Bedding/Comforters & Sets
80753,AmazonBasics,An Amazon Brand,NaN,AmazonBasics Reversible Microfiber Comforter B...,NaN,NaN,HOME_BED_AND_BATH,B00U8QGU32,US,PrimeNow,81wIj3ibqJL,/Categories
133319,AmazonBasics,Apple MFi certified charging and syncing cable...,Black,AmazonBasics Lightning to USB A Cable - MFi Ce...,NaN,apple | 6 | 6s | 5 | 5s | plus | power | data ...,WIRELESS_ACCESSORY,B017YEANB0,US,Amazon,71OAJcL9oKL,/Categories/Accessories/Cables & Adapters


In [ ]:
item_cntry_grp[item_cntry_grp['Count']!=2]

,item_id,Count
40,B00EZ1ZTFG,3
82,B00TIFV5BG,3
84,B00U8QGU32,3
106,B017YEANB0,3
140,B01MUG3BAM,3
168,B01NGTPJKB,3
205,B06XK8DNLF,3
332,B0788FT982,3
368,B07CTL3BX6,3
381,B07CTNGHM3,3


In [ ]:
removed_df['item_id'].nunique()

707

In [ ]:
non_english_contries = ['DE','MX',	'ES',	'FR',	'IT','JP',	'NL',	'SE',	'SA',	'TR',	'PL','BR']
NE_removed_df = removed_df[~removed_df['country'].isin(non_english_contries)]
NE_removed_df['item_id'].shape

# Only two Item id different after non_english_contries filter

(1298,)

In [ ]:
NE_cntry_grp = NE_removed_df.groupby('item_id').agg(Count = ('country','count')).reset_index()
NE_cntry_grp[NE_cntry_grp['Count']>2]

# NE_removed_df[NE_removed_df['item_id'].isin(NE_cntry_grp.loc[NE_cntry_grp['Count']!=2,'item_id'])].sort_values(by='item_id')

,item_id,Count
82,B00TIFV5BG,3
140,B01MUG3BAM,3
205,B06XK8DNLF,3
330,B0788FT982,3
366,B07CTL3BX6,3
443,B07K4SL4HQ,3
484,B07NC3S9PK,3
526,B07RRWD659,3
566,B07W5R75YP,3
569,B07W6W46BQ,3


In [ ]:
removed_df.sort_values(by='item_id').head(20)

,brand,bullet_point,color,item_name,model_name,item_keywords,product_type,item_id,country,marketplace,main_image_id,node
39670,Pinzon by Amazon,An Amazon Brand,White,Pinzon 400-Thread-Count Hemstitch Egyptian Cot...,NaN,bedding | sheets,HOME,B000GWJZPS,US,PrimeNow,91Akl1oo8hL,/Categories/Bedding/Sheets & Pillowcases/Sheet...
54581,Pinzon by Amazon,An Amazon Brand,White,Pinzon Hemstitch 400-Thread-Count 100 Percent ...,NaN,bedding; sheets | bedding | sheets | bedding |...,HOME,B000GWJZPS,AU,Amazon,61KXh-+pFYL,/Categories/Bedding & Linen/Sheets & Pillowcas...
1228,Pinzon by Amazon,NaN,NaN,Pinzon Hemstitch 400-Thread-Count 100 Percent ...,NaN,NaN,HOME,B000ON030U,CA,Amazon,71dxkG6bZPL,/Categories/Home Textiles/Bedding & Linen/Shee...
80188,Unknown,NaN,COLOR_NAME,Pinzon Hemstitch 400-Thread-Count 100 Percent ...,NaN,bedding,HOME,B000ON030U,US,PrimeNow,717KxNq7PrL,/Categories/Bedding/Sheets & Pillowcases/Pillo...
90424,Pinzon by Amazon,White down pillow with removable cover offered...,White,Pinzon Pyrenees Hypoallergenic White Down Pillow,NaN,bedding,HOME,B000W3YW5Y,SG,Amazon,815ig6jBv4S,/Homeware & Furniture/Bedding/Pillows
104979,Pinzon by Amazon,White down pillow with removable cover offered...,White,Pinzon Pyrenees Hypoallergenic White Down Pillow,NaN,bedding,HOME,B000W3YW5Y,GB,Amazon,815ig6jBv4S,/Home & Garden/Home & Kitchen/Categories/Beddi...
145687,Amazon Essentials,Pendant necklace featuring a sparkling round c...,Platinum Plated Sterling Silver,Amazon Essentials Platinum Plated Sterling Sil...,NaN,necklaces | sparkly | solitiare | CZ | CZ jewe...,FINENECKLACEBRACELETANKLET,B0015MN8KG,US,Amazon,710Ny5F0peL,/Departments/Women/Jewelry/Necklaces/Pendant N...
4813,Amazon Essentials,Pendant necklace featuring a sparkling round c...,Platinum Plated Sterling Silver,Amazon Essentials Platinum Plated Sterling Sil...,NaN,necklaces | sparkly | solitiare | CZ | CZ jewe...,FINENECKLACEBRACELETANKLET,B0015MN8KG,AE,Amazon,71UszmFEevL,/Categories/Women/Jewelry/Necklaces
4269,Amazon Essentials,Classic stud earrings featuring cubic zirconia...,Platinum Plated Sterling Silver,Amazon Essentials Platinum Plated Sterling Sil...,NaN,earring | studs | post | posts | solitaire | C...,FINEEARRING,B0015MSD2O,AE,Amazon,61IdZ8R33uL,/Categories/Women/Jewelry/Earrings
132315,Amazon Essentials,Classic stud earrings featuring cubic zirconia...,Platinum Plated Sterling Silver,Amazon Essentials Platinum Plated Sterling Sil...,NaN,earring | studs | post | posts | solitaire | C...,FINEEARRING,B0015MSD2O,US,Amazon,710OSAy3O8L,/Departments/Women/Jewelry/Earrings/Stud


In [ ]:
brands_df.head()

,brand,bullet_point,color,item_name,model_name,item_keywords,product_type,item_id,country,marketplace,main_image_id,node
2,AmazonBasics,3D printer filament with 1.75mm diameter + / -...,Translucent Yellow,"AmazonBasics PETG 3D Printer Filament, 1.75mm,...",NaN,3d printer filament | petg printer filament | ...,MECHANICAL_COMPONENTS,B07H9GMYXS,AE,Amazon,81NP7qh2L6L,/Categories
3,Stone & Beam,NaN,Stone Brown,"Stone & Beam Stone Brown Swatch, 25020039-01",NaN,love | loveseat | queen | for | couch | cheste...,SOFA,B07CTPR73M,GB,Amazon,61Rp4qOih9L,/Home & Garden/Home & Kitchen/Categories/Furni...
4,The Fix,Embroidered flowers bloom against understated ...,Havana Tan,The Fix Amazon Brand Women's French Floral Emb...,NaN,zapatos shoe para de ladies mujer womans mocas...,SHOES,B01MTEI8M6,AU,Amazon,714CmIfKIYL,/Categories/Women/Shoes/Loafer Flats
5,Amazon Brand - Solimo,"Snug fit for Mi Redmi Go, with perfect cut-out...",Multicolor,Amazon Brand - Solimo Designer Autumn Girl 3D ...,Mi Redmi Go,cellphonecover | backcase | mobileguard | mobi...,CELLULAR_PHONE_CASE,B0853X2F4M,IN,Amazon,81+4dBN1jsL,/Categories/Mobiles & Accessories/Mobile Acces...
6,Amazon Brand - Solimo,"Snug fit for Xiaomi Redmi Y2, with perfect cut...",multi-colored,Amazon Brand - Solimo Designer Butterflies Pri...,Xiaomi Redmi Y2,Xiaomi Redmi Y2 Mobile back case cover transpa...,CELLULAR_PHONE_CASE,B07R91S92W,IN,Amazon,61LWeNhjZ9L,/Categories/Mobiles & Accessories/Mobile Acces...


In [ ]:
brands_df.isna().sum()/brands_df.shape[0]

,0
brand,0.033820
bullet_point,0.075428
color,0.197337
item_name,0.000041
model_name,0.391603
item_keywords,0.123620
product_type,0.000000
item_id,0.000000
country,0.000000
marketplace,0.000000


In [ ]:
with pd.option_context('display.max_colwidth', None):
    print(brands_df[['item_name','bullet_point','item_keywords']].sample(1))

                                                                                                  item_name  \
126730  Amazon Brand - Solimo Designer Lake View UV Printed Soft Back Case Mobile Cover for Mi Redmi Note 4   

                                                                                                                                                                                                                                                                                                                                                                                                                                          bullet_point  \
126730  Snug fit for Mi Redmi Note 4, with perfect cut-outs for volume buttons, audio and charging ports | Compatible with Mi Redmi Note 4 | Easy to put & take off with perfect cutouts for volume buttons, audio & charging ports. | Stylish design and appearance, express your unique personality. | Extreme precision design allows easy a

Based on what we are seeing with the keywords values we need to first identify if the item keywords column offers anything extra than item_name + bullet_point.

for each listing, take the set of unique words in item_keywords and the set of unique words in your combined_text (item_name + bullet_point), and compute what fraction of item_keywords' unique vocabulary is already covered by combined_text.


In [ ]:
import re
import numpy as np

text_df = brands_df.copy()
text_df = text_df[['item_name','bullet_point','item_keywords']]
text_df['combined_text'] = text_df['item_name'] + " " + text_df['bullet_point']


def clean_words(text):
    if not isinstance(text, str) or text.strip() == '':
        return set()
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)  # strip punctuation, keep word chars
    return set(w for w in text.split() if w)

text_df['combined_words'] = text_df['combined_text'].apply(clean_words)
text_df['item_keywords_words'] = text_df['item_keywords'].apply(clean_words)

def overlap_ratio(row):
    kw = row['item_keywords_words']
    if len(kw) == 0:
        return np.nan  # no keywords present — exclude from the metric, don't count as 0
    return len(kw & row['combined_words']) / len(kw)

text_df['overlap_ratio'] = text_df.apply(overlap_ratio, axis=1)
text_df['overlap_ratio'].describe()

,overlap_ratio
count,107566.000000
mean,0.525976
std,0.257965
min,0.000000
25%,0.360000
50%,0.625000
75%,0.714286
max,1.000000


In [ ]:
with pd.option_context('display.max_colwidth', None):
  print(text_df[['combined_words','item_keywords_words','overlap_ratio']].sample(1))

                                                                                                                                                                                                                                                                                                                                                                                                          combined_words  \
42972  {life, put, hard, raised, cutouts, perfect, bezel, camera, printed, extreme, charging, while, vivo, slim, access, head, audio, phone, personality, your, buttons, precision, take, for, solimo, to, flat, screen, express, featuring, volume, surface, with, stylish, 3d, all, no, and, unique, off, design, appearance, cover, amazon, y51l, warranty, easy, mobile, case, allows, designer, brand, back, ports}   

                                            item_keywords_words  overlap_ratio  
42972  {vivo, panel, cover, phone, oneplus, mobile, case, back}           0.75  


After looking at the overlap ratio, decide not to include the keywords, for now
let's just move forward with the combined text and then later we can also include keywords and can draw comparision between both the approaches. Because the cost of tokens will be higher and additional information is not that much great. Most keywords are used just for the SEO purposes.


In [ ]:
brands_df.shape

(113198, 11)

In [ ]:
imaged_duplicated = brands_df[brands_df['main_image_id'].duplicated()]

In [ ]:
without_cellular = brands_df[brands_df['product_type'] != 'CELLULAR_PHONE_CASE']
without_cellular['main_image_id'].duplicated().sum()

np.int64(0)

In [ ]:
image_grouped = brands_df.groupby('main_image_id').agg(Count = ('main_image_id','count'))

image_dup = image_grouped[image_grouped['Count']>1].index

image_duplicated = brands_df[brands_df['main_image_id'].isin(image_dup)].sort_values(by='main_image_id')
image_duplicated.head(20)

,brand,color,model_name,item_keywords,product_type,item_id,country,marketplace,main_image_id,node,combined_text,ct_len,country_rank


In [ ]:
image_duplicated.shape

(15999, 11)

The final set is at 100k rows with 60k being cellular phone case. Considering that this ABO dataset does not give us the idea distribution or real world product distribution on amazon we can either choose to use the same 100k dataset or take a subset of this dataset. If we exlcude the 60k. We are left with 40k rows data where the product distribution is quite better than before and if we go ahead and take a subset of these by excluding the product with very less counts (not sure what is the need to exclude them) we could get a reasonable dataset to use further in the project.

From the analysis I can see that we can not exclude the product type with lesser number of count as their distribution is quite large.

From this point what we can do is that take 5-6 categories and make a project on them, or take all except the top 1-2 and then make project based on generelisation.


In [ ]:
imaged_duplicated['product_type'].value_counts().head(20)

,count
product_type,
SHOES,3723
GROCERY,1198
HOME,512
SANDAL,406
BOOT,405
CELLULAR_PHONE_CASE,399
FINERING,342
HEALTH_PERSONAL_CARE,176
HOME_BED_AND_BATH,157


In [ ]:
with pd.option_context('display.max_colwidth', None):
  print(product_df[product_df['count']<10].sample(1))

     product_type                                            node  count
2686      GROCERY  /Categories/Produce/Fresh Vegetables/Asparagus      2


,product_type,node,count
0,CELLULAR_PHONE_CASE,/Categories/Mobiles & Accessories/Mobile Acces...,44363
1,CELLULAR_PHONE_CASE,/Categories/Mobiles & Accessories/Mobile Acces...,18660
2,CELLULAR_PHONE_CASE,/Categories/Mobiles & Accessories/Mobile Acces...,903
3,SHOES,/Categories/Shoes/Men's Shoes/Casual Shoes/Sne...,818
4,SOFA,/Categories/Furniture/Living Room Furniture/So...,722
5,CHAIR,/Categories/Furniture/Living Room Furniture/Ch...,668
6,SHOES,/Categories/Shoes/Men's Shoes/Formal Shoes,657
7,SHOES,/Categories/Shoes/Men's Shoes/Sports & Outdoor...,397
8,HANDBAG,"/Categories/Handbags, Purses & Clutches/Handba...",354
9,STOOL_SEATING,/Categories/Furniture/Game & Recreation Room F...,306


In [ ]:
product_df = brands_df['product_type'].value_counts().reset_index()

In [ ]:
product_df # 504 products

product_df[product_df['count']>1] # 440 products
product_df[product_df['count']>5] # 323 products
product_df[product_df['count']>10] # 257 products
product_df[product_df['count']>50] # 114 products
product_df[product_df['count']>70] # 86 products
product_df[product_df['count']>80] # 80 products
product_df[product_df['count']>100] # 69 products
product_df[product_df['count']>200] # 35 products
product_df[product_df['count']>300] # 29 products # 300 would be a good number
# to be used as lower cap, will give good amount of samples for train test and val
product_df[product_df['count']>500] # 13 products
product_df[product_df['count']>1000] # 4 products

product_df

,product_type,count
0,CELLULAR_PHONE_CASE,64323
1,SHOES,4814
2,GROCERY,4612
3,HOME,1530
4,CHAIR,1386
...,...,...
499,JEWELRY,1
500,DRILL,1
501,GARLIC_PRESS,1
502,WIRELESS_LOCKED_PHONE,1


In [ ]:
new_product_df = product_df.loc[(product_df['count']>300) & (product_df['product_type']!='CELLULAR_PHONE_CASE'),'product_type']
new_product_df
brands_df.head()
brands_df[brands_df['product_type'].isin(new_product_df)] # 25479 rows
# SHOES	4814
# GROCERY	4612
# HOME	1530
# CHAIR	1386

,brand,color,model_name,item_keywords,product_type,item_id,country,marketplace,main_image_id,node,combined_text,ct_len,country_rank
4,The Fix,Havana Tan,NaN,zapatos shoe para de ladies mujer womans mocas...,SHOES,B01MTEI8M6,AU,Amazon,714CmIfKIYL,/Categories/Women/Shoes/Loafer Flats,The Fix Amazon Brand Women's French Floral Emb...,226,10
35,Stanton,Brown,NaN,Stanton | Brown | 8 UK (42 EU) (9 US) | FK/YT1...,SHOES,B0846BTJ34,IN,Amazon,71kE0ihplPL,/Categories/Shoes/Men's Shoes/Casual Shoes/Loa...,Stanton Men's Brown Loafers-8 UK (42 EU) (9 US...,191,0
39,The Fix,Taupe,NaN,zapatos shoe para de ladies mujer womans | des...,SHOES,B01N27SMXC,US,Amazon,71-eMu0gN+L,/Departments/Women/Shops,Amazon Brand - The Fix Women's Foley Tassel Sl...,199,1
50,206 Collective,Black Suede,NaN,fathers day gifts for dad | party | ladies pum...,SHOES,B01N5XROXR,AE,Amazon,61uONZVpAEL,/Categories/Men/Shoes/Loafer Flats,Amazon Brand - 206 Collective Men's Pike Drivi...,195,13
55,365 Everyday Value,NaN,NaN,NaN,GROCERY,B07QC4DZ2L,US,WholeFoods,81oNfRT6ebL,"/Categories/Dairy, Cheese & Eggs/Milk & Cream/...","365 EVERYDAY VALUE Organic Light Cream, 1 PT B...",435,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
147666,365 Everyday Value,NaN,NaN,NaN,GROCERY,B07SY3YFC1,US,WholeFoods,71MgCNGNF+L,NaN,"365 EVERYDAY VALUE Original Oatmilk, 64 FZ Bro...",576,1
147671,AmazonBasics,Cool White,NaN,This bulb is energy star rated and UL listed |...,LIGHT_BULB,B079VNBKJB,US,Amazon,71oRw-drx5L,/Categories/Light Bulbs/LED Bulbs,"AmazonBasics Commercial Grade 25,000 Hour LED ...",893,1
147685,Flavia,GREY,NaN,NaN,SHOES,B0845BV6H6,IN,Amazon,811AS57BWVL,/Categories/Shoes/Women's Shoes/Sports & Outdo...,FLAVIA Women's Grey Running Shoes-UK6 (FKT/SD-...,187,0
147686,Unknown,Opulence Grey,NaN,Pinzon by Amazon | Supima cotton;bathroom;towe...,HOME,B003JFJWVG,CA,Amazon,91IqWKj2vQL,NaN,"Pinzon Oversized Supima Cotton Wash Cloth, Opp...",363,2


In [ ]:
final_counts = brands_df[brands_df['product_type'].isin(new_product_df)]['product_type'].value_counts()
final_counts
print(f"top category share: {final_counts.iloc[0] / final_counts.sum():.1%}")
print(f"top-5 share: {final_counts.iloc[:5].sum() / final_counts.sum():.1%}")

top category share: 18.9%
top-5 share: 52.3%


In [ ]:
final_counts

,count
product_type,
SHOES,4814
GROCERY,4612
HOME,1530
CHAIR,1386
HOME_FURNITURE_AND_DECOR,989
HEALTH_PERSONAL_CARE,926
SOFA,825
FINENECKLACEBRACELETANKLET,816
RUG,760


In [ ]:
image_df = pd.read_csv(f'{BASE}/images_meta/images.csv.gz', compression='gzip')
image_df.head()

,image_id,height,width,path
0,010-mllS7JL,106,106,14/14fe8812.jpg
1,01dkn0Gyx0L,122,122,da/daab0cad.jpg
2,01sUPg0387L,111,111,d2/d2daaae9.jpg
3,1168jc-5r1L,186,186,3a/3a4e88e6.jpg
4,11RUV5Fs65L,30,500,d9/d91ab9cf.jpg


In [ ]:
!aws s3 ls --no-sign-request s3://amazon-berkeley-objects/images/small/

                           PRE 00/
                           PRE 01/
                           PRE 02/
                           PRE 03/
                           PRE 04/
                           PRE 05/
                           PRE 06/
                           PRE 07/
                           PRE 08/
                           PRE 09/
                           PRE 0a/
                           PRE 0b/
                           PRE 0c/
                           PRE 0d/
                           PRE 0e/
                           PRE 0f/
                           PRE 10/
                           PRE 11/
                           PRE 12/
                           PRE 13/
                           PRE 14/
                           PRE 15/
                           PRE 16/
                           PRE 17/
                           PRE 18/
                           PRE 19/
                           PRE 1a/
                           PRE 1b/
                    

In [ ]:
text = '''                           PRE 00/
                           PRE 01/
                           PRE 02/
                           PRE 03/
                           PRE 04/
                           PRE 05/
                           PRE 06/
                           PRE 07/
                           PRE 08/
                           PRE 09/
                           PRE 0a/
                           PRE 0b/
                           PRE 0c/
                           PRE 0d/
                           PRE 0e/
                           PRE 0f/
                           PRE 10/
                           PRE 11/
                           PRE 12/
                           PRE 13/
                           PRE 14/
                           PRE 15/
                           PRE 16/
                           PRE 17/
                           PRE 18/
                           PRE 19/
                           PRE 1a/
                           PRE 1b/
                           PRE 1c/
                           PRE 1d/
                           PRE 1e/
                           PRE 1f/
                           PRE 20/
                           PRE 21/
                           PRE 22/
                           PRE 23/
                           PRE 24/
                           PRE 25/
                           PRE 26/
                           PRE 27/
                           PRE 28/
                           PRE 29/
                           PRE 2a/
                           PRE 2b/
                           PRE 2c/
                           PRE 2d/
                           PRE 2e/
                           PRE 2f/
                           PRE 30/
                           PRE 31/
                           PRE 32/
                           PRE 33/
                           PRE 34/
                           PRE 35/
                           PRE 36/
                           PRE 37/
                           PRE 38/
                           PRE 39/
                           PRE 3a/
                           PRE 3b/
                           PRE 3c/
                           PRE 3d/
                           PRE 3e/
                           PRE 3f/
                           PRE 40/
                           PRE 41/
                           PRE 42/
                           PRE 43/
                           PRE 44/
                           PRE 45/
                           PRE 46/
                           PRE 47/
                           PRE 48/
                           PRE 49/
                           PRE 4a/
                           PRE 4b/
                           PRE 4c/
                           PRE 4d/
                           PRE 4e/
                           PRE 4f/
                           PRE 50/
                           PRE 51/
                           PRE 52/
                           PRE 53/
                           PRE 54/
                           PRE 55/
                           PRE 56/
                           PRE 57/
                           PRE 58/
                           PRE 59/
                           PRE 5a/
                           PRE 5b/
                           PRE 5c/
                           PRE 5d/
                           PRE 5e/
                           PRE 5f/
                           PRE 60/
                           PRE 61/
                           PRE 62/
                           PRE 63/
                           PRE 64/
                           PRE 65/
                           PRE 66/
                           PRE 67/
                           PRE 68/
                           PRE 69/
                           PRE 6a/
                           PRE 6b/
                           PRE 6c/
                           PRE 6d/
                           PRE 6e/
                           PRE 6f/
                           PRE 70/
                           PRE 71/
                           PRE 72/
                           PRE 73/
                           PRE 74/
                           PRE 75/
                           PRE 76/
                           PRE 77/
                           PRE 78/
                           PRE 79/
                           PRE 7a/
                           PRE 7b/
                           PRE 7c/
                           PRE 7d/
                           PRE 7e/
                           PRE 7f/
                           PRE 80/
                           PRE 81/
                           PRE 82/
                           PRE 83/
                           PRE 84/
                           PRE 85/
                           PRE 86/
                           PRE 87/
                           PRE 88/
                           PRE 89/
                           PRE 8a/
                           PRE 8b/
                           PRE 8c/
                           PRE 8d/
                           PRE 8e/
                           PRE 8f/
                           PRE 90/
                           PRE 91/
                           PRE 92/
                           PRE 93/
                           PRE 94/
                           PRE 95/
                           PRE 96/
                           PRE 97/
                           PRE 98/
                           PRE 99/
                           PRE 9a/
                           PRE 9b/
                           PRE 9c/
                           PRE 9d/
                           PRE 9e/
                           PRE 9f/
                           PRE a0/
                           PRE a1/
                           PRE a2/
                           PRE a3/
                           PRE a4/
                           PRE a5/
                           PRE a6/
                           PRE a7/
                           PRE a8/
                           PRE a9/
                           PRE aa/
                           PRE ab/
                           PRE ac/
                           PRE ad/
                           PRE ae/
                           PRE af/
                           PRE b0/
                           PRE b1/
                           PRE b2/
                           PRE b3/
                           PRE b4/
                           PRE b5/
                           PRE b6/
                           PRE b7/
                           PRE b8/
                           PRE b9/
                           PRE ba/
                           PRE bb/
                           PRE bc/
                           PRE bd/
                           PRE be/
                           PRE bf/
                           PRE c0/
                           PRE c1/
                           PRE c2/
                           PRE c3/
                           PRE c4/
                           PRE c5/
                           PRE c6/
                           PRE c7/
                           PRE c8/
                           PRE c9/
                           PRE ca/
                           PRE cb/
                           PRE cc/
                           PRE cd/
                           PRE ce/
                           PRE cf/
                           PRE d0/
                           PRE d1/
                           PRE d2/
                           PRE d3/
                           PRE d4/
                           PRE d5/
                           PRE d6/
                           PRE d7/
                           PRE d8/
                           PRE d9/
                           PRE da/
                           PRE db/
                           PRE dc/
                           PRE dd/
                           PRE de/
                           PRE df/
                           PRE e0/
                           PRE e1/
                           PRE e2/
                           PRE e3/
                           PRE e4/
                           PRE e5/
                           PRE e6/
                           PRE e7/
                           PRE e8/
                           PRE e9/
                           PRE ea/
                           PRE eb/
                           PRE ec/
                           PRE ed/
                           PRE ee/
                           PRE ef/
                           PRE f0/
                           PRE f1/
                           PRE f2/
                           PRE f3/
                           PRE f4/
                           PRE f5/
                           PRE f6/
                           PRE f7/
                           PRE f8/
                           PRE f9/
                           PRE fa/
                           PRE fb/
                           PRE fc/
                           PRE fd/
                           PRE fe/
                           PRE ff/'''

In [ ]:
text_split = text.replace("\n","").replace("PRE","'").replace("/","', ").replace(" ","")

text_split

"'00','01','02','03','04','05','06','07','08','09','0a','0b','0c','0d','0e','0f','10','11','12','13','14','15','16','17','18','19','1a','1b','1c','1d','1e','1f','20','21','22','23','24','25','26','27','28','29','2a','2b','2c','2d','2e','2f','30','31','32','33','34','35','36','37','38','39','3a','3b','3c','3d','3e','3f','40','41','42','43','44','45','46','47','48','49','4a','4b','4c','4d','4e','4f','50','51','52','53','54','55','56','57','58','59','5a','5b','5c','5d','5e','5f','60','61','62','63','64','65','66','67','68','69','6a','6b','6c','6d','6e','6f','70','71','72','73','74','75','76','77','78','79','7a','7b','7c','7d','7e','7f','80','81','82','83','84','85','86','87','88','89','8a','8b','8c','8d','8e','8f','90','91','92','93','94','95','96','97','98','99','9a','9b','9c','9d','9e','9f','a0','a1','a2','a3','a4','a5','a6','a7','a8','a9','aa','ab','ac','ad','ae','af','b0','b1','b2','b3','b4','b5','b6','b7','b8','b9','ba','bb','bc','bd','be','bf','c0','c1','c2','c3','c4','c5','c6','c7'

,image_id,height,width,path,path_map
0,010-mllS7JL,106,106,14/14fe8812.jpg,{'14/14fe8812.jpg': '/content/drive/MyDrive/Im...
1,01dkn0Gyx0L,122,122,da/daab0cad.jpg,{'da/daab0cad.jpg': '/content/drive/MyDrive/Im...
2,01sUPg0387L,111,111,d2/d2daaae9.jpg,{'d2/d2daaae9.jpg': '/content/drive/MyDrive/Im...
3,1168jc-5r1L,186,186,3a/3a4e88e6.jpg,{'3a/3a4e88e6.jpg': '/content/drive/MyDrive/Im...
4,11RUV5Fs65L,30,500,d9/d91ab9cf.jpg,{'d9/d91ab9cf.jpg': '/content/drive/MyDrive/Im...
...,...,...,...,...,...
398207,B1zv8OpTkBS,2560,2560,6d/6d49d130.jpg,{'6d/6d49d130.jpg': '/content/drive/MyDrive/Im...
398208,B1zwflWhPIS,2560,2560,b1/b163e0ea.jpg,{'b1/b163e0ea.jpg': '/content/drive/MyDrive/Im...
398209,C1lf45DhhRS,2560,2560,a1/a116d9d1.jpg,{'a1/a116d9d1.jpg': '/content/drive/MyDrive/Im...
398210,C1pEt6jBLiS,2560,2560,9c/9c3e1158.jpg,{'9c/9c3e1158.jpg': '/content/drive/MyDrive/Im...


In [ ]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.9 MB/s eta 0:00:00


In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from concurrent.futures import ThreadPoolExecutor, as_completed

s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
BUCKET = 'amazon-berkeley-objects'
local_base = f'{BASE}/small_images'

def download_image_if_missing(image_path, local_path):
    if os.path.exists(local_path):
        return local_path, True  # already have it
    try:
        s3.download_file(BUCKET, f'images/small/{image_path}', local_path)
        return local_path, True
    except Exception as e:
        print(f"FAILED: {image_path} -> {e}")
        return local_path, False

def download_images_parallel(path_map, max_workers=20):
    """path_map: dict of {image_path: local_path}"""
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(download_image_if_missing, p, lp): p
                   for p, lp in path_map.items()}
        for future in as_completed(futures):
            path = futures[future]
            local_path, success = future.result()
            results[path] = success
    return results

In [ ]:
download_if_missing_image_file('s3://amazon-berkeley-objects/images/metadata/images.csv.gz', f'{BASE}/images_meta/images.csv.gz')
image_df = images_gz_extract('images.csv.gz')
image_df['path_map'] = image_df['path'].apply(lambda x:f'{BASE}/small_images/{x}')

brands_img = brands_df.merge(image_df, left_on='main_image_id', right_on = 'image_id')
brands_img = brands_img[brands_img['image_id']!= None]

image_map = brands_img.set_index('path')['path_map'].to_dict()

already have /content/drive/MyDrive/Image-Text Retrieval/images_meta/images.csv.gz, skipping


In [ ]:
with pd.option_context('display.max_colwidth', None):
  print(brands_img['path_map'].sample(1))

7092    {'b8/b845dd83.jpg': '/content/drive/MyDrive/Image-Text Retrieval/small_images/b8/b845dd83.jpg'}
Name: path_map, dtype: object


In [ ]:
download_image_if_missing('b8/b845dd83.jpg','/content/drive/MyDrive/Image-Text Retrieval/small_images/')

('/content/drive/MyDrive/Image-Text Retrieval/small_images/', True)

In [ ]:
sample_image_map = brands_img.set_index('path')['path_map'].sample(10).to_dict()

In [ ]:
doanload_images = download_images_parallel(image_map)

In [ ]:
download_images = pd.DataFrame(download_images)


{'2b/2bca3877.jpg': True,
 'f5/f5c61542.jpg': True,
 'b8/b84e832b.jpg': True,
 'fb/fb47073f.jpg': True,
 '80/800631ba.jpg': True,
 '98/98c9fe5e.jpg': True,
 '52/52396ece.jpg': True,
 '5b/5bad4f84.jpg': True,
 'c4/c4d9a09c.jpg': True,
 'f1/f1f8c6ac.jpg': True}

In [ ]:
brands_img.shape

(25479, 18)

In [ ]:

# Move to retreiving the images for the dataset. to see if we can get enough or not it all is good

# move towards the EDA of the dataset.
# Next decisions will be based on EDA